In [ ]:
# V7 Cell 1 — Spin-Dependent Admissibility Experiment

import numpy as np
import pandas as pd

print("=" * 70)
print("V7 — SPIN-DEPENDENT ADMISSIBILITY")
print("=" * 70)

# --------------------------------------------------
# Research question
# --------------------------------------------------

RESEARCH_QUESTION = """
How does spin magnitude and spin axis orientation
reshape the physically admissible region of a legal
tennis serve?
"""

print("\nResearch question:")
print(RESEARCH_QUESTION)


# --------------------------------------------------
# Fixed baseline conditions
# --------------------------------------------------

BASE_SPEED_KMH = 200.0
CONTACT_HEIGHT_M = 3.0

TARGET_SIDE = "deuce"

DRAG_COEFFICIENT = 0.55
AIR_DENSITY = 1.21

print("Baseline conditions:")
print(f"  Serve speed:       {BASE_SPEED_KMH:.1f} km/h")
print(f"  Contact height:    {CONTACT_HEIGHT_M:.2f} m")
print(f"  Target:            {TARGET_SIDE}")
print(f"  Drag coefficient:  {DRAG_COEFFICIENT:.2f}")
print(f"  Air density:       {AIR_DENSITY:.2f} kg/m³")


# --------------------------------------------------
# Spin experiment
# --------------------------------------------------

# Spin rates are specified in revolutions per minute.
SPIN_RATES_RPM = np.array([
    0,
    500,
    1000,
    1500,
    2000,
    2500
], dtype=float)

print("\nVertical-spin experiment:")
print(
    "  Spin axis: +y "
    "(Magnus force primarily affects vertical motion)"
)

print(
    f"  Spin rates: {SPIN_RATES_RPM.astype(int).tolist()} rpm"
)


# --------------------------------------------------
# Convert rpm to angular velocity
# --------------------------------------------------

# 1 revolution = 2π radians
# 1 minute = 60 seconds

SPIN_RATES_RAD_S = (
    SPIN_RATES_RPM
    * 2.0
    * np.pi
    / 60.0
)


# --------------------------------------------------
# Construct spin vectors
# --------------------------------------------------

vertical_spin_vectors = np.array([
    [0.0, omega, 0.0]
    for omega in SPIN_RATES_RAD_S
])


# --------------------------------------------------
# Experiment table
# --------------------------------------------------

spin_experiment_df = pd.DataFrame({
    "spin_rpm": SPIN_RATES_RPM,
    "spin_rad_s": SPIN_RATES_RAD_S,
    "omega_x": vertical_spin_vectors[:, 0],
    "omega_y": vertical_spin_vectors[:, 1],
    "omega_z": vertical_spin_vectors[:, 2]
})


print("\nExperiment table:")
print(spin_experiment_df.to_string(index=False))


# --------------------------------------------------
# Hypotheses
# --------------------------------------------------

print("\nHypotheses:")
print("  H1: Vertical spin changes the net-clearance boundary.")
print("  H2: Vertical spin changes the service-line boundary.")
print("  H3: Vertical spin changes angular admissibility width.")
print("  H4: Pure vertical spin produces negligible direct")
print("      lateral Magnus displacement.")
print()
print("V7 Cell 1: EXPERIMENT DEFINITION READY")

In [ ]:
# V7 Cell 2 — Verified 3D Serve Physics Model

import numpy as np
from scipy.integrate import solve_ivp

print("=" * 70)
print("V7 — VERIFIED 3D SERVE PHYSICS MODEL")
print("=" * 70)


# --------------------------------------------------
# Physical constants
# --------------------------------------------------

G = 9.81
BALL_MASS = 0.0575
BALL_DIAMETER = 0.067
BALL_RADIUS = BALL_DIAMETER / 2.0

AIR_DENSITY = 1.21
DRAG_COEFFICIENT = 0.55

BALL_AREA = np.pi * BALL_RADIUS**2

# Provisional spin parameter from V3–V6
V_SPIN = 20.0


# --------------------------------------------------
# Court geometry
# --------------------------------------------------

NET_X = 0.0
SERVICE_LINE_X = 6.40

BASELINE_X = 11.885
SERVER_X = -BASELINE_X

SERVICE_BOX_WIDTH = 4.115

NET_HEIGHT_CENTER = 0.914
NET_HEIGHT_POST = 1.07

CONTACT_HEIGHT = 3.0


# --------------------------------------------------
# Lift coefficient
# --------------------------------------------------

def lift_coefficient(speed, spin_speed):
    """
    Provisional lift-coefficient model.

    spin_speed = R * |omega|
    """

    if speed <= 0 or spin_speed <= 0:
        return 0.0

    return 1.0 / (
        2.0 + speed / spin_speed
    )


# --------------------------------------------------
# Magnus acceleration
# --------------------------------------------------

def magnus_acceleration(
    velocity,
    omega
):
    velocity = np.asarray(
        velocity,
        dtype=float
    )

    omega = np.asarray(
        omega,
        dtype=float
    )

    speed = np.linalg.norm(
        velocity
    )

    spin_rate = np.linalg.norm(
        omega
    )

    if speed == 0 or spin_rate == 0:
        return np.zeros(3)

    velocity_hat = (
        velocity / speed
    )

    omega_hat = (
        omega / spin_rate
    )

    spin_speed = (
        BALL_RADIUS * spin_rate
    )

    C_L = lift_coefficient(
        speed,
        spin_speed
    )

    magnus_direction = np.cross(
        omega_hat,
        velocity_hat
    )

    force_magnitude = (
        0.5
        * AIR_DENSITY
        * BALL_AREA
        * C_L
        * speed**2
    )

    force = (
        force_magnitude
        * magnus_direction
    )

    return force / BALL_MASS


# --------------------------------------------------
# Full acceleration model
# --------------------------------------------------

def serve_acceleration(
    velocity,
    omega
):
    velocity = np.asarray(
        velocity,
        dtype=float
    )

    speed = np.linalg.norm(
        velocity
    )

    # Drag
    if speed > 0:

        drag_factor = (
            -0.5
            * AIR_DENSITY
            * DRAG_COEFFICIENT
            * BALL_AREA
            * speed
            / BALL_MASS
        )

        a_drag = (
            drag_factor * velocity
        )

    else:

        a_drag = np.zeros(3)


    # Magnus
    a_magnus = magnus_acceleration(
        velocity,
        omega
    )


    # Gravity
    a_gravity = np.array([
        0.0,
        0.0,
        -G
    ])


    return (
        a_gravity
        + a_drag
        + a_magnus
    )


# --------------------------------------------------
# 3D trajectory
# --------------------------------------------------

def serve_trajectory(
    t,
    state,
    omega
):

    x, y, z, vx, vy, vz = state

    velocity = np.array([
        vx,
        vy,
        vz
    ])

    acceleration = serve_acceleration(
        velocity,
        omega
    )

    return np.array([
        vx,
        vy,
        vz,
        acceleration[0],
        acceleration[1],
        acceleration[2]
    ])


# --------------------------------------------------
# Events
# --------------------------------------------------

def net_event(t, state):

    return state[0] - NET_X


net_event.direction = 1


def ground_event(t, state):

    return state[2]


ground_event.terminal = True
ground_event.direction = -1


# --------------------------------------------------
# Serve simulation
# --------------------------------------------------

def simulate_serve_v7(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    omega
):

    speed = speed_kmh / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )


    # Initial velocity
    initial_velocity = np.array([
        speed
        * np.cos(theta)
        * np.cos(phi),

        speed
        * np.cos(theta)
        * np.sin(phi),

        speed
        * np.sin(theta)
    ])


    # Initial state
    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT,

        initial_velocity[0],
        initial_velocity[1],
        initial_velocity[2]
    ])


    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),

        t_span=(0.0, 4.0),

        y0=initial_state,

        events=[
            net_event,
            ground_event
        ],

        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,

        dense_output=True
    )

    return solution


# --------------------------------------------------
# Extract events
# --------------------------------------------------

def extract_serve_events(solution):

    net_state = None
    landing_state = None

    if len(solution.t_events[0]) > 0:

        net_time = (
            solution.t_events[0][0]
        )

        net_state = (
            solution.sol(net_time)
        )


    if len(solution.t_events[1]) > 0:

        landing_time = (
            solution.t_events[1][0]
        )

        landing_state = (
            solution.sol(landing_time)
        )


    return (
        net_state,
        landing_state
    )


# --------------------------------------------------
# Court geometry
# --------------------------------------------------

def net_height(y):

    y = np.asarray(
        y
    )

    return (
        NET_HEIGHT_CENTER
        + (
            NET_HEIGHT_POST
            - NET_HEIGHT_CENTER
        )
        * np.minimum(
            np.abs(y)
            / SERVICE_BOX_WIDTH,
            1.0
        )
    )


def net_clearance(
    z,
    y
):

    return (
        z
        - net_height(y)
    )


# --------------------------------------------------
# Model summary
# --------------------------------------------------

print("Physical model:")
print(f"  Ball mass:          {BALL_MASS:.4f} kg")
print(f"  Ball diameter:      {BALL_DIAMETER:.4f} m")
print(f"  Air density:        {AIR_DENSITY:.2f} kg/m³")
print(f"  Drag coefficient:   {DRAG_COEFFICIENT:.2f}")
print(f"  Provisional V_SPIN: {V_SPIN:.1f} m/s")

print("\nCourt model:")
print(f"  Server x:            {SERVER_X:.3f} m")
print(f"  Net x:               {NET_X:.3f} m")
print(f"  Service line x:      {SERVICE_LINE_X:.3f} m")
print(f"  Service half-width:  {SERVICE_BOX_WIDTH:.3f} m")

print("\nV7 Cell 2: 3D PHYSICS MODEL READY")

In [ ]:
# V7 Cell 3 — Regression Against V6 Zero-Spin Baseline

print("=" * 70)
print("V7 — REGRESSION AGAINST V6")
print("=" * 70)

# --------------------------------------------------
# V6 reference case
# --------------------------------------------------

TEST_SPEED_KMH = 200.0
TEST_ANGLE_DEG = -7.914436
TEST_AZIMUTH_DEG = 0.0

ZERO_SPIN = np.array([
    0.0,
    0.0,
    0.0
])


# --------------------------------------------------
# V6 verified reference values
# --------------------------------------------------

V6_LANDING_X = 5.282063
V6_NET_CLEARANCE = 0.162464


# --------------------------------------------------
# Run V7 simulation
# --------------------------------------------------

solution = simulate_serve_v7(
    speed_kmh=TEST_SPEED_KMH,
    launch_angle_deg=TEST_ANGLE_DEG,
    azimuth_deg=TEST_AZIMUTH_DEG,
    omega=ZERO_SPIN
)

net_state, landing_state = extract_serve_events(
    solution
)


# --------------------------------------------------
# Extract V7 results
# --------------------------------------------------

v7_net_y = net_state[1]
v7_net_z = net_state[2]

v7_landing_x = landing_state[0]
v7_landing_y = landing_state[1]

v7_clearance = net_clearance(
    v7_net_z,
    v7_net_y
)


# --------------------------------------------------
# Calculate differences
# --------------------------------------------------

landing_x_error = abs(
    v7_landing_x
    - V6_LANDING_X
)

clearance_error = abs(
    v7_clearance
    - V6_NET_CLEARANCE
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print("\nV6 reference:")
print(
    f"  Landing x:       "
    f"{V6_LANDING_X:.6f} m"
)

print(
    f"  Net clearance:   "
    f"{V6_NET_CLEARANCE:.6f} m"
)

print("\nV7 result:")
print(
    f"  Landing x:       "
    f"{v7_landing_x:.6f} m"
)

print(
    f"  Landing y:       "
    f"{v7_landing_y:.6f} m"
)

print(
    f"  Net clearance:   "
    f"{v7_clearance:.6f} m"
)

print("\nRegression errors:")
print(
    f"  Landing x error: "
    f"{landing_x_error:.3e} m"
)

print(
    f"  Clearance error: "
    f"{clearance_error:.3e} m"
)


# --------------------------------------------------
# Pass/fail criteria
# --------------------------------------------------

landing_pass = (
    landing_x_error < 1e-4
)

clearance_pass = (
    clearance_error < 1e-4
)

print("\n" + "-" * 70)

print(
    "Landing position regression: "
    f"{'PASS' if landing_pass else 'FAIL'}"
)

print(
    "Net clearance regression:    "
    f"{'PASS' if clearance_pass else 'FAIL'}"
)


if landing_pass and clearance_pass:

    print("\nV7 REGRESSION: PASS")

else:

    print("\nV7 REGRESSION: FAIL")

print("=" * 70)

In [ ]:
# V7 Cell 4 — Vertical Magnus Direction Verification

print("=" * 70)
print("V7 — VERTICAL MAGNUS DIRECTION VERIFICATION")
print("=" * 70)


# --------------------------------------------------
# Test conditions
# --------------------------------------------------

TEST_SPEED_KMH = 200.0
TEST_ANGLE_DEG = -7.914436
TEST_AZIMUTH_DEG = 0.0

SPIN_TEST_RPM = 1500.0

SPIN_TEST_RAD_S = (
    SPIN_TEST_RPM
    * 2.0
    * np.pi
    / 60.0
)


# Three spin states
spin_cases = {
    "No spin": np.array([
        0.0, 0.0, 0.0
    ]),

    "+1500 rpm": np.array([
        0.0,
        SPIN_TEST_RAD_S,
        0.0
    ]),

    "-1500 rpm": np.array([
        0.0,
        -SPIN_TEST_RAD_S,
        0.0
    ])
}


# --------------------------------------------------
# Run simulations
# --------------------------------------------------

results = []

for label, omega in spin_cases.items():

    solution = simulate_serve_v7(
        speed_kmh=TEST_SPEED_KMH,
        launch_angle_deg=TEST_ANGLE_DEG,
        azimuth_deg=TEST_AZIMUTH_DEG,
        omega=omega
    )

    net_state, landing_state = (
        extract_serve_events(solution)
    )

    net_y = net_state[1]
    net_z = net_state[2]

    landing_x = landing_state[0]
    landing_y = landing_state[1]

    clearance = net_clearance(
        net_z,
        net_y
    )

    results.append({
        "spin_case": label,
        "spin_rpm":
            0.0 if label == "No spin"
            else (
                SPIN_TEST_RPM
                if label == "+1500 rpm"
                else -SPIN_TEST_RPM
            ),
        "net_z_m": net_z,
        "net_clearance_m": clearance,
        "landing_x_m": landing_x,
        "landing_y_m": landing_y
    })


spin_direction_df = pd.DataFrame(
    results
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print("\nTest conditions:")
print(
    f"  Speed:       {TEST_SPEED_KMH:.1f} km/h"
)

print(
    f"  Angle:       {TEST_ANGLE_DEG:.6f}°"
)

print(
    f"  Azimuth:     {TEST_AZIMUTH_DEG:.1f}°"
)

print(
    f"  Spin magnitude: {SPIN_TEST_RPM:.0f} rpm"
)

print("\nResults:")
print(
    spin_direction_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# --------------------------------------------------
# Direction checks
# --------------------------------------------------

no_spin = spin_direction_df.iloc[0]
positive_spin = spin_direction_df.iloc[1]
negative_spin = spin_direction_df.iloc[2]

positive_delta_z = (
    positive_spin["net_z_m"]
    - no_spin["net_z_m"]
)

negative_delta_z = (
    negative_spin["net_z_m"]
    - no_spin["net_z_m"]
)

lateral_max = max(
    abs(positive_spin["landing_y_m"]),
    abs(negative_spin["landing_y_m"])
)


# --------------------------------------------------
# Verification
# --------------------------------------------------

opposite_direction = (
    positive_delta_z
    * negative_delta_z
    < 0
)

negligible_lateral = (
    lateral_max < 1e-6
)

spin_changes_trajectory = (
    abs(positive_delta_z) > 1e-5
    and abs(negative_delta_z) > 1e-5
)


print("\n" + "-" * 70)
print("PHYSICAL-DIRECTION CHECKS")
print("-" * 70)

print(
    "Positive and negative spin produce "
    "opposite vertical effects: "
    f"{'PASS' if opposite_direction else 'FAIL'}"
)

print(
    "Vertical spin produces negligible "
    "lateral displacement: "
    f"{'PASS' if negligible_lateral else 'FAIL'}"
)

print(
    "Spin measurably changes trajectory: "
    f"{'PASS' if spin_changes_trajectory else 'FAIL'}"
)


overall_pass = (
    opposite_direction
    and negligible_lateral
    and spin_changes_trajectory
)

print("\n" + "=" * 70)

if overall_pass:
    print("V7 MAGNUS DIRECTION VERIFICATION: PASS")
else:
    print("V7 MAGNUS DIRECTION VERIFICATION: FAIL")

print("=" * 70)

In [ ]:
# V7 Cell 5 — Fixed-Launch Spin Response

print("=" * 70)
print("V7 — FIXED-LAUNCH SPIN RESPONSE")
print("=" * 70)

# --------------------------------------------------
# Fixed launch condition
# --------------------------------------------------

TEST_SPEED_KMH = 200.0
TEST_ANGLE_DEG = -7.914436
TEST_AZIMUTH_DEG = 0.0


# --------------------------------------------------
# Spin sweep
# --------------------------------------------------

SPIN_SWEEP_RPM = np.arange(
    -2500.0,
    2500.0 + 250.0,
    250.0
)


results = []


# --------------------------------------------------
# Run simulations
# --------------------------------------------------

for spin_rpm in SPIN_SWEEP_RPM:

    spin_rad_s = (
        spin_rpm
        * 2.0
        * np.pi
        / 60.0
    )

    omega = np.array([
        0.0,
        spin_rad_s,
        0.0
    ])

    solution = simulate_serve_v7(
        speed_kmh=TEST_SPEED_KMH,
        launch_angle_deg=TEST_ANGLE_DEG,
        azimuth_deg=TEST_AZIMUTH_DEG,
        omega=omega
    )

    net_state, landing_state = (
        extract_serve_events(solution)
    )

    net_y = net_state[1]
    net_z = net_state[2]

    landing_x = landing_state[0]

    clearance = net_clearance(
        net_z,
        net_y
    )

    # Ground-event time
    landing_time = (
        solution.t_events[1][0]
    )

    results.append({
        "spin_rpm": spin_rpm,
        "spin_rad_s": spin_rad_s,
        "net_clearance_m": clearance,
        "landing_x_m": landing_x,
        "landing_time_s": landing_time
    })


spin_response_df = pd.DataFrame(
    results
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print("\nFixed launch condition:")
print(
    f"  Speed:   {TEST_SPEED_KMH:.1f} km/h"
)

print(
    f"  Angle:   {TEST_ANGLE_DEG:.6f}°"
)

print(
    f"  Azimuth: {TEST_AZIMUTH_DEG:.1f}°"
)

print("\nSpin response:")
print(
    spin_response_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# --------------------------------------------------
# Monotonicity checks
# --------------------------------------------------

clearances = (
    spin_response_df[
        "net_clearance_m"
    ].values
)

landing_distances = (
    spin_response_df[
        "landing_x_m"
    ].values
)


clearance_decreases = np.all(
    np.diff(clearances) < 0
)

landing_decreases = np.all(
    np.diff(landing_distances) < 0
)


# --------------------------------------------------
# Zero-spin regression
# --------------------------------------------------

zero_row = spin_response_df[
    np.isclose(
        spin_response_df["spin_rpm"],
        0.0
    )
].iloc[0]

zero_spin_landing_error = abs(
    zero_row["landing_x_m"]
    - 5.282063
)

zero_spin_clearance_error = abs(
    zero_row["net_clearance_m"]
    - 0.162464
)


print("\n" + "-" * 70)
print("RESPONSE CHECKS")
print("-" * 70)

print(
    "Net clearance decreases with increasing "
    f"spin: {'PASS' if clearance_decreases else 'FAIL'}"
)

print(
    "Landing distance decreases with increasing "
    f"spin: {'PASS' if landing_decreases else 'FAIL'}"
)

print(
    f"Zero-spin landing regression error: "
    f"{zero_spin_landing_error:.3e} m"
)

print(
    f"Zero-spin clearance regression error: "
    f"{zero_spin_clearance_error:.3e} m"
)


overall_pass = (
    clearance_decreases
    and landing_decreases
    and zero_spin_landing_error < 1e-4
    and zero_spin_clearance_error < 1e-4
)


print("\n" + "=" * 70)

if overall_pass:
    print("V7 FIXED-LAUNCH SPIN RESPONSE: PASS")
else:
    print("V7 FIXED-LAUNCH SPIN RESPONSE: REVIEW REQUIRED")

print("=" * 70)

In [ ]:
# V7 Cell 6 — Corrected Fast Spin-Dependent 1D Boundary Continuation

from scipy.optimize import brentq

print("=" * 70)
print("V7 — CORRECTED SPIN-DEPENDENT 1D ADMISSIBILITY")
print("=" * 70)


# --------------------------------------------------
# Fixed conditions
# --------------------------------------------------

SPEED_KMH = 200.0
AZIMUTH_DEG = 0.0

SPIN_RATES_RPM = np.array([
    -2500,
    -2000,
    -1500,
    -1000,
    -500,
    0,
    500,
    1000,
    1500,
    2000,
    2500
], dtype=float)


# --------------------------------------------------
# Spin vector
# --------------------------------------------------

def omega_from_rpm(spin_rpm):

    return np.array([
        0.0,
        spin_rpm * 2.0 * np.pi / 60.0,
        0.0
    ])


# --------------------------------------------------
# Boundary-safe trajectory
# --------------------------------------------------

def get_boundary_quantities(
    angle_deg,
    spin_rpm
):

    omega = omega_from_rpm(
        spin_rpm
    )

    speed = SPEED_KMH / 3.6

    theta = np.radians(
        angle_deg
    )

    phi = np.radians(
        AZIMUTH_DEG
    )


    # Initial velocity
    initial_velocity = np.array([
        speed * np.cos(theta) * np.cos(phi),
        speed * np.cos(theta) * np.sin(phi),
        speed * np.sin(theta)
    ])


    # Initial state
    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT,
        initial_velocity[0],
        initial_velocity[1],
        initial_velocity[2]
    ])


    # --------------------------------------------------
    # Integrate through net
    # --------------------------------------------------

    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),

        t_span=(0.0, 2.0),

        y0=initial_state,

        events=net_event,

        rtol=1e-9,
        atol=1e-11,
        max_step=0.002,

        dense_output=True
    )


    if len(solution.t_events[0]) == 0:
        return None


    net_time = solution.t_events[0][0]

    net_state = solution.sol(
        net_time
    )


    # Net clearance
    clearance = net_clearance(
        net_state[2],
        net_state[1]
    )


    # --------------------------------------------------
    # Continue from net until ground
    # --------------------------------------------------

    ground_solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),

        t_span=(
            net_time,
            2.0
        ),

        y0=net_state,

        events=ground_event,

        rtol=1e-9,
        atol=1e-11,
        max_step=0.002,

        dense_output=True
    )


    if len(
        ground_solution.t_events[0]
    ) == 0:

        landing_x = np.nan

    else:

        landing_time = (
            ground_solution.t_events[0][0]
        )

        landing_state = (
            ground_solution.sol(
                landing_time
            )
        )

        landing_x = landing_state[0]


    return {
        "net_clearance": clearance,
        "landing_x": landing_x
    }


# --------------------------------------------------
# Root functions
# --------------------------------------------------

def net_root(
    angle_deg,
    spin_rpm
):

    result = get_boundary_quantities(
        angle_deg,
        spin_rpm
    )

    if result is None:
        return np.nan

    return result["net_clearance"]


def service_root(
    angle_deg,
    spin_rpm
):

    result = get_boundary_quantities(
        angle_deg,
        spin_rpm
    )

    if result is None:
        return np.nan

    return (
        result["landing_x"]
        - SERVICE_LINE_X
    )


# --------------------------------------------------
# Local bracket finder
# --------------------------------------------------

def find_local_bracket(
    function,
    center,
    spin_rpm,
    half_width=0.8,
    step=0.2
):

    angles = np.arange(
        center - half_width,
        center + half_width + step,
        step
    )

    previous_angle = None
    previous_value = None

    for angle in angles:

        value = function(
            angle,
            spin_rpm
        )

        if not np.isfinite(value):

            previous_angle = None
            previous_value = None
            continue


        if (
            previous_value is not None
            and previous_value * value <= 0
        ):

            return (
                previous_angle,
                angle
            )


        previous_angle = angle
        previous_value = value


    return None


# --------------------------------------------------
# Known zero-spin boundaries
# --------------------------------------------------

ZERO_SPIN_NET = -8.675886
ZERO_SPIN_SERVICE = -7.152987


boundary_cache = {
    0: {
        "net": ZERO_SPIN_NET,
        "service": ZERO_SPIN_SERVICE
    }
}


# --------------------------------------------------
# Continuation order
# --------------------------------------------------

ordered_spins = [
    0,
    500,
    1000,
    1500,
    2000,
    2500,
    -500,
    -1000,
    -1500,
    -2000,
    -2500
]


results = []


# --------------------------------------------------
# Continue boundaries through spin
# --------------------------------------------------

for spin_rpm in ordered_spins:

    if spin_rpm == 0:
        continue


    if spin_rpm > 0:
        reference_spin = spin_rpm - 500
    else:
        reference_spin = spin_rpm + 500


    reference = boundary_cache[
        reference_spin
    ]


    # --------------------------------------------------
    # Net boundary
    # --------------------------------------------------

    net_bracket = find_local_bracket(
        net_root,
        reference["net"],
        spin_rpm
    )


    if net_bracket is None:

        net_angle = np.nan

    else:

        net_angle = brentq(
            lambda angle:
                net_root(
                    angle,
                    spin_rpm
                ),
            net_bracket[0],
            net_bracket[1],
            xtol=1e-7
        )


    # --------------------------------------------------
    # Service-line boundary
    # --------------------------------------------------

    service_bracket = find_local_bracket(
        service_root,
        reference["service"],
        spin_rpm
    )


    if service_bracket is None:

        service_angle = np.nan

    else:

        service_angle = brentq(
            lambda angle:
                service_root(
                    angle,
                    spin_rpm
                ),
            service_bracket[0],
            service_bracket[1],
            xtol=1e-7
        )


    # Cache results
    boundary_cache[spin_rpm] = {
        "net": net_angle,
        "service": service_angle
    }


    # Angular width
    if (
        np.isfinite(net_angle)
        and np.isfinite(service_angle)
    ):

        width = (
            service_angle
            - net_angle
        )

    else:

        width = np.nan


    results.append({
        "spin_rpm": spin_rpm,
        "net_boundary_deg": net_angle,
        "service_boundary_deg": service_angle,
        "angle_width_deg": width
    })


# --------------------------------------------------
# Add zero-spin baseline
# --------------------------------------------------

results.append({
    "spin_rpm": 0.0,
    "net_boundary_deg": ZERO_SPIN_NET,
    "service_boundary_deg": ZERO_SPIN_SERVICE,
    "angle_width_deg":
        ZERO_SPIN_SERVICE
        - ZERO_SPIN_NET
})


# --------------------------------------------------
# Final table
# --------------------------------------------------

spin_boundary_df = (
    pd.DataFrame(results)
    .sort_values("spin_rpm")
    .reset_index(drop=True)
)


print("\nSpin-dependent admissibility:")
print(
    spin_boundary_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


print("\n" + "=" * 70)
print("V7 Cell 6: CORRECTED BOUNDARY SWEEP COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 7 — Spin Boundary Validation

print("=" * 70)
print("V7 — SPIN BOUNDARY VALIDATION")
print("=" * 70)


# --------------------------------------------------
# Extract arrays
# --------------------------------------------------

spin_values = (
    spin_boundary_df["spin_rpm"].values
)

net_boundaries = (
    spin_boundary_df["net_boundary_deg"].values
)

service_boundaries = (
    spin_boundary_df[
        "service_boundary_deg"
    ].values
)

widths = (
    spin_boundary_df[
        "angle_width_deg"
    ].values
)


# --------------------------------------------------
# Basic validity
# --------------------------------------------------

all_finite = (
    np.isfinite(net_boundaries).all()
    and np.isfinite(service_boundaries).all()
    and np.isfinite(widths).all()
)

all_positive_width = np.all(
    widths > 0
)


# --------------------------------------------------
# Monotonicity
# --------------------------------------------------

net_monotonic = np.all(
    np.diff(net_boundaries) > 0
)

service_monotonic = np.all(
    np.diff(service_boundaries) > 0
)

width_monotonic = np.all(
    np.diff(widths) > 0
)


# --------------------------------------------------
# Zero-spin regression
# --------------------------------------------------

zero_index = np.where(
    np.isclose(
        spin_values,
        0.0
    )
)[0][0]


zero_net_error = abs(
    net_boundaries[zero_index]
    - (-8.675886)
)

zero_service_error = abs(
    service_boundaries[zero_index]
    - (-7.152987)
)

zero_width_error = abs(
    widths[zero_index]
    - 1.522899
)


zero_spin_pass = (
    zero_net_error < 1e-4
    and zero_service_error < 1e-4
    and zero_width_error < 1e-4
)


# --------------------------------------------------
# Print checks
# --------------------------------------------------

print("\nValidation checks:")
print("-" * 70)

print(
    "All boundary values finite: "
    f"{'PASS' if all_finite else 'FAIL'}"
)

print(
    "All angular widths positive: "
    f"{'PASS' if all_positive_width else 'FAIL'}"
)

print(
    "Net boundary increases monotonically "
    "with spin: "
    f"{'PASS' if net_monotonic else 'FAIL'}"
)

print(
    "Service boundary increases monotonically "
    "with spin: "
    f"{'PASS' if service_monotonic else 'FAIL'}"
)

print(
    "Angular width increases monotonically "
    "with spin: "
    f"{'PASS' if width_monotonic else 'FAIL'}"
)

print(
    "Zero-spin regression: "
    f"{'PASS' if zero_spin_pass else 'FAIL'}"
)


# --------------------------------------------------
# Overall result
# --------------------------------------------------

overall_pass = (
    all_finite
    and all_positive_width
    and net_monotonic
    and service_monotonic
    and width_monotonic
    and zero_spin_pass
)


print("\n" + "=" * 70)

if overall_pass:
    print("V7 1D SPIN VALIDATION: PASS")
else:
    print("V7 1D SPIN VALIDATION: REVIEW REQUIRED")

print("=" * 70)

In [ ]:
# V7 Cell 8 — Spin Sensitivity Analysis

print("=" * 70)
print("V7 — SPIN SENSITIVITY ANALYSIS")
print("=" * 70)


# --------------------------------------------------
# Extract data
# --------------------------------------------------

spin = spin_boundary_df[
    "spin_rpm"
].values

net_boundary = spin_boundary_df[
    "net_boundary_deg"
].values

service_boundary = spin_boundary_df[
    "service_boundary_deg"
].values

width = spin_boundary_df[
    "angle_width_deg"
].values


# --------------------------------------------------
# Finite-difference sensitivity
# --------------------------------------------------

net_sensitivity = np.gradient(
    net_boundary,
    spin
)

service_sensitivity = np.gradient(
    service_boundary,
    spin
)

width_sensitivity = np.gradient(
    width,
    spin
)


# --------------------------------------------------
# Zero-spin reference
# --------------------------------------------------

zero_idx = np.where(
    np.isclose(spin, 0.0)
)[0][0]

zero_width = width[
    zero_idx
]

zero_net = net_boundary[
    zero_idx
]

zero_service = service_boundary[
    zero_idx
]


# --------------------------------------------------
# Percentage change
# --------------------------------------------------

width_percent_change = (
    (width - zero_width)
    / zero_width
    * 100.0
)


# --------------------------------------------------
# Build analysis table
# --------------------------------------------------

spin_sensitivity_df = pd.DataFrame({

    "spin_rpm":
        spin,

    "net_boundary_deg":
        net_boundary,

    "service_boundary_deg":
        service_boundary,

    "angle_width_deg":
        width,

    "net_sensitivity_deg_per_1000rpm":
        net_sensitivity * 1000.0,

    "service_sensitivity_deg_per_1000rpm":
        service_sensitivity * 1000.0,

    "width_sensitivity_deg_per_1000rpm":
        width_sensitivity * 1000.0,

    "width_change_from_zero_percent":
        width_percent_change
})


# --------------------------------------------------
# Output
# --------------------------------------------------

print("\nSpin sensitivity table:")
print(
    spin_sensitivity_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# --------------------------------------------------
# Summary statistics
# --------------------------------------------------

negative_width = width[
    spin < 0
]

positive_width = width[
    spin > 0
]

negative_spin_min = np.min(
    negative_width
)

positive_spin_max = np.max(
    positive_width
)


total_width_change = (
    positive_spin_max
    - negative_spin_min
)

total_width_percent = (
    total_width_change
    / negative_spin_min
    * 100.0
)


print("\n" + "-" * 70)
print("SPIN SENSITIVITY SUMMARY")
print("-" * 70)

print(
    f"Zero-spin angular width: "
    f"{zero_width:.6f}°"
)

print(
    f"Width at -2500 rpm:     "
    f"{negative_spin_min:.6f}°"
)

print(
    f"Width at +2500 rpm:     "
    f"{positive_spin_max:.6f}°"
)

print(
    f"Total width change:     "
    f"{total_width_change:.6f}°"
)

print(
    f"Relative change:        "
    f"{total_width_percent:.2f}%"
)


# --------------------------------------------------
# Local sensitivity around zero spin
# --------------------------------------------------

zero_plus_idx = np.where(
    np.isclose(spin, 500.0)
)[0][0]

zero_minus_idx = np.where(
    np.isclose(spin, -500.0)
)[0][0]


local_width_sensitivity = (
    width[zero_plus_idx]
    - width[zero_minus_idx]
) / 1000.0


local_net_sensitivity = (
    net_boundary[zero_plus_idx]
    - net_boundary[zero_minus_idx]
) / 1000.0


local_service_sensitivity = (
    service_boundary[zero_plus_idx]
    - service_boundary[zero_minus_idx]
) / 1000.0


print("\nLocal sensitivity around zero spin:")
print(
    f"  Net boundary:     "
    f"{local_net_sensitivity:.6f}° / 1000 rpm"
)

print(
    f"  Service boundary: "
    f"{local_service_sensitivity:.6f}° / 1000 rpm"
)

print(
    f"  Angular width:    "
    f"{local_width_sensitivity:.6f}° / 1000 rpm"
)


print("\n" + "=" * 70)
print("V7 Cell 8: SPIN SENSITIVITY ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 9 — Spin-Dependent Boundary Visualization

import matplotlib.pyplot as plt

print("=" * 70)
print("V7 — SPIN-DEPENDENT BOUNDARY VISUALIZATION")
print("=" * 70)


# --------------------------------------------------
# Extract data
# --------------------------------------------------

spin = spin_boundary_df["spin_rpm"].values

net_boundary = (
    spin_boundary_df[
        "net_boundary_deg"
    ].values
)

service_boundary = (
    spin_boundary_df[
        "service_boundary_deg"
    ].values
)

width = (
    spin_boundary_df[
        "angle_width_deg"
    ].values
)


# --------------------------------------------------
# Figure 1 — Net boundary
# --------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    spin,
    net_boundary,
    marker="o"
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Spin rate (rpm)")
plt.ylabel("Net boundary angle (degrees)")
plt.title(
    "Net-Clearance Boundary vs Spin Rate"
)

plt.grid(True, alpha=0.3)

plt.show()


# --------------------------------------------------
# Figure 2 — Service boundary
# --------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    spin,
    service_boundary,
    marker="o"
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Spin rate (rpm)")
plt.ylabel(
    "Service-line boundary angle (degrees)"
)

plt.title(
    "Service-Line Boundary vs Spin Rate"
)

plt.grid(True, alpha=0.3)

plt.show()


# --------------------------------------------------
# Figure 3 — Angular admissibility width
# --------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    spin,
    width,
    marker="o"
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Spin rate (rpm)")
plt.ylabel(
    "Admissible launch-angle width (degrees)"
)

plt.title(
    "Admissible Launch-Angle Width vs Spin Rate"
)

plt.grid(True, alpha=0.3)

plt.show()


# --------------------------------------------------
# Quantitative summary
# --------------------------------------------------

print("\nKey values:")

print(
    f"  Width at -2500 rpm: "
    f"{width[0]:.6f}°"
)

print(
    f"  Width at 0 rpm:     "
    f"{width[len(width)//2]:.6f}°"
)

print(
    f"  Width at +2500 rpm: "
    f"{width[-1]:.6f}°"
)

print(
    f"  Total width change: "
    f"{width[-1] - width[0]:.6f}°"
)

print("\n" + "=" * 70)
print("V7 Cell 9: VISUALIZATION COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 10 — Nonlinear Spin-Response Analysis

from numpy.polynomial.polynomial import polyfit, polyval

print("=" * 70)
print("V7 — NONLINEAR SPIN-RESPONSE ANALYSIS")
print("=" * 70)


# --------------------------------------------------
# Data
# --------------------------------------------------

spin = spin_boundary_df["spin_rpm"].values

net_boundary = (
    spin_boundary_df[
        "net_boundary_deg"
    ].values
)

service_boundary = (
    spin_boundary_df[
        "service_boundary_deg"
    ].values
)

width = (
    spin_boundary_df[
        "angle_width_deg"
    ].values
)


# --------------------------------------------------
# R² helper
# --------------------------------------------------

def calculate_r2(y, y_pred):

    ss_res = np.sum(
        (y - y_pred) ** 2
    )

    ss_tot = np.sum(
        (y - np.mean(y)) ** 2
    )

    return (
        1.0
        - ss_res / ss_tot
    )


# --------------------------------------------------
# Fit linear and quadratic models
# --------------------------------------------------

quantities = {
    "Net boundary": net_boundary,
    "Service boundary": service_boundary,
    "Angular width": width
}


fit_results = {}


for name, values in quantities.items():

    linear_coeff = polyfit(
        spin,
        values,
        1
    )

    quadratic_coeff = polyfit(
        spin,
        values,
        2
    )

    linear_pred = polyval(
        spin,
        linear_coeff
    )

    quadratic_pred = polyval(
        spin,
        quadratic_coeff
    )

    linear_r2 = calculate_r2(
        values,
        linear_pred
    )

    quadratic_r2 = calculate_r2(
        values,
        quadratic_pred
    )

    fit_results[name] = {
        "linear_coeff": linear_coeff,
        "quadratic_coeff": quadratic_coeff,
        "linear_pred": linear_pred,
        "quadratic_pred": quadratic_pred,
        "linear_r2": linear_r2,
        "quadratic_r2": quadratic_r2
    }


# --------------------------------------------------
# Print results
# --------------------------------------------------

for name in quantities:

    result = fit_results[name]

    linear = result["linear_coeff"]
    quadratic = result["quadratic_coeff"]

    print("\n" + "-" * 70)
    print(name.upper())
    print("-" * 70)

    print(
        "Linear model:"
    )

    print(
        f"  y(s) = "
        f"{linear[0]:.8f} "
        f"+ ({linear[1]:.10f})s"
    )

    print(
        f"  R² = "
        f"{result['linear_r2']:.8f}"
    )

    print(
        "\nQuadratic model:"
    )

    print(
        f"  y(s) = "
        f"{quadratic[0]:.8f} "
        f"+ ({quadratic[1]:.10f})s "
        f"+ ({quadratic[2]:.12f})s²"
    )

    print(
        f"  R² = "
        f"{result['quadratic_r2']:.8f}"
    )

    print(
        f"  R² improvement = "
        f"{result['quadratic_r2'] - result['linear_r2']:.8f}"
    )


# --------------------------------------------------
# Width residuals
# --------------------------------------------------

width_quadratic_pred = fit_results[
    "Angular width"
]["quadratic_pred"]

width_residuals = (
    width
    - width_quadratic_pred
)

width_max_residual = np.max(
    np.abs(width_residuals)
)

width_rms_residual = np.sqrt(
    np.mean(width_residuals ** 2)
)


# --------------------------------------------------
# Print width residual diagnostics
# --------------------------------------------------

print("\n" + "=" * 70)
print("ANGULAR-WIDTH RESIDUAL DIAGNOSTICS")
print("=" * 70)

print(
    f"Maximum absolute residual: "
    f"{width_max_residual:.8f}°"
)

print(
    f"RMS residual:              "
    f"{width_rms_residual:.8f}°"
)

print("\nV7 Cell 10: NONLINEAR SPIN ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 11 — Spin-Reversal Symmetry Analysis

print("=" * 70)
print("V7 — SPIN-REVERSAL SYMMETRY ANALYSIS")
print("=" * 70)


# --------------------------------------------------
# Select symmetric spin pairs
# --------------------------------------------------

test_spins = [-2500, -1500, 0, 1500, 2500]

subset = spin_boundary_df[
    spin_boundary_df["spin_rpm"].isin(test_spins)
].copy()

subset = subset.sort_values("spin_rpm")


# --------------------------------------------------
# Extract values
# --------------------------------------------------

values = {
    row["spin_rpm"]: {
        "net": row["net_boundary_deg"],
        "service": row["service_boundary_deg"],
        "width": row["angle_width_deg"]
    }
    for _, row in subset.iterrows()
}


zero_width = values[0]["width"]
zero_net = values[0]["net"]
zero_service = values[0]["service"]


# --------------------------------------------------
# Symmetry calculations
# --------------------------------------------------

print("\nSpin-response symmetry:")

symmetry_results = []

for magnitude in [1500, 2500]:

    negative = values[-magnitude]
    positive = values[magnitude]

    net_neg_change = negative["net"] - zero_net
    net_pos_change = positive["net"] - zero_net

    service_neg_change = negative["service"] - zero_service
    service_pos_change = positive["service"] - zero_service

    width_neg_change = negative["width"] - zero_width
    width_pos_change = positive["width"] - zero_width

    net_symmetry_error = (
        net_pos_change + net_neg_change
    )

    service_symmetry_error = (
        service_pos_change + service_neg_change
    )

    width_symmetry_error = (
        width_pos_change + width_neg_change
    )

    print("\n" + "-" * 70)
    print(f"±{magnitude} rpm")
    print("-" * 70)

    print(
        f"Net boundary change:"
        f"  -spin = {net_neg_change:+.6f}°"
        f"  +spin = {net_pos_change:+.6f}°"
    )

    print(
        f"Net symmetry error:"
        f" {net_symmetry_error:+.6f}°"
    )

    print(
        f"Service boundary change:"
        f"  -spin = {service_neg_change:+.6f}°"
        f"  +spin = {service_pos_change:+.6f}°"
    )

    print(
        f"Service symmetry error:"
        f" {service_symmetry_error:+.6f}°"
    )

    print(
        f"Width change:"
        f"  -spin = {width_neg_change:+.6f}°"
        f"  +spin = {width_pos_change:+.6f}°"
    )

    print(
        f"Width symmetry error:"
        f" {width_symmetry_error:+.6f}°"
    )

    symmetry_results.append({
        "spin_magnitude_rpm": magnitude,
        "net_symmetry_error_deg": net_symmetry_error,
        "service_symmetry_error_deg": service_symmetry_error,
        "width_symmetry_error_deg": width_symmetry_error
    })


# --------------------------------------------------
# Interpretation
# --------------------------------------------------

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

print(
    """
A perfectly odd response around zero spin would have
equal-and-opposite boundary changes.

Small symmetry errors indicate that the response is
approximately symmetric over the tested spin range.

The test is treated as a model diagnostic, not as a
requirement that real tennis aerodynamics be perfectly
symmetric.
"""
)


# --------------------------------------------------
# Automated check
# --------------------------------------------------

max_error = max(
    max(abs(r["net_symmetry_error_deg"]),
        abs(r["service_symmetry_error_deg"]),
        abs(r["width_symmetry_error_deg"]))
    for r in symmetry_results
)

print(
    f"Maximum symmetry error: "
    f"{max_error:.6f}°"
)

if max_error < 0.10:
    print(
        "Spin-reversal response is approximately symmetric: PASS"
    )
else:
    print(
        "Spin-reversal response shows substantial asymmetry: REVIEW"
    )

print("\nV7 Cell 11: SPIN-REVERSAL ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 12 — Spin × Azimuth Admissibility Envelope

from scipy.optimize import root
import numpy as np

print("=" * 70)
print("V7 — SPIN × AZIMUTH ADMISSIBILITY ENVELOPE")
print("=" * 70)


# --------------------------------------------------
# Configuration
# --------------------------------------------------

SPEED_KMH = 200.0
CONTACT_HEIGHT = 3.0

SPIN_VALUES_RPM = np.array([
    -2500,
    -2000,
    -1500,
    -1000,
    -500,
    0,
    500,
    1000,
    1500,
    2000,
    2500
], dtype=float)


# --------------------------------------------------
# Boundary-safe trajectory evaluation
# --------------------------------------------------

def evaluate_endpoint_v7(
    launch_angle_deg,
    azimuth_deg,
    spin_rpm
):
    """
    Evaluate net clearance and landing position.

    The trajectory is first integrated to the net.
    It is then continued to ground so that the actual
    landing position is obtained.

    This function is intended for boundary solving.
    """

    speed_ms = SPEED_KMH / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )

    vx0 = (
        speed_ms
        * np.cos(theta)
        * np.cos(phi)
    )

    vy0 = (
        speed_ms
        * np.cos(theta)
        * np.sin(phi)
    )

    vz0 = (
        speed_ms
        * np.sin(theta)
    )

    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT,
        vx0,
        vy0,
        vz0
    ])

    omega = np.array([
        0.0,
        np.radians(spin_rpm / 60.0),
        0.0
    ])

    # ----------------------------------------------
    # Integrate to net
    # ----------------------------------------------

    def net_event(t, state):
        return state[0] - NET_X

    net_event.terminal = True
    net_event.direction = 1

    solution_net = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),
        (0.0, 2.0),
        initial_state,
        events=net_event,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.002
    )

    if len(solution_net.t_events[0]) == 0:
        return None

    net_state = solution_net.y_events[0][0]

    net_y = net_state[1]
    net_z = net_state[2]

    clearance = (
        net_z
        - net_height(net_y)
    )

    # ----------------------------------------------
    # Continue from net to ground
    # ----------------------------------------------

    def ground_event(t, state):
        return state[2]

    ground_event.terminal = True
    ground_event.direction = -1

    solution_ground = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),
        (
            solution_net.t_events[0][0],
            2.0
        ),
        net_state,
        events=ground_event,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.002
    )

    if len(solution_ground.t_events[0]) == 0:
        return None

    landing_state = (
        solution_ground.y_events[0][0]
    )

    landing_x = landing_state[0]
    landing_y = landing_state[1]

    return {
        "clearance": clearance,
        "landing_x": landing_x,
        "landing_y": landing_y
    }


# --------------------------------------------------
# Solve simultaneous outer-boundary conditions
# --------------------------------------------------

def endpoint_residuals_v7(
    variables,
    spin_rpm
):
    """
    Outer lateral endpoint:

        1. Net clearance = 0
        2. Landing y = service-box boundary
    """

    angle_deg, azimuth_deg = variables

    result = evaluate_endpoint_v7(
        angle_deg,
        azimuth_deg,
        spin_rpm
    )

    if result is None:
        return [
            100.0,
            100.0
        ]

    return [
        result["clearance"],
        result["landing_y"] - SERVICE_BOX_WIDTH
    ]


# --------------------------------------------------
# Continuation through spin
# --------------------------------------------------

results = []

# Starting point from the previously verified
# zero-spin endpoint.
previous_guess = np.array([
    -7.84590700,
    13.78989255
])


for spin_rpm in SPIN_VALUES_RPM:

    solution = root(
        lambda variables:
            endpoint_residuals_v7(
                variables,
                spin_rpm
            ),
        previous_guess,
        method="hybr",
        options={
            "xtol": 1e-8
        }
    )

    if not solution.success:
        print(
            f"WARNING: root solver did not fully converge "
            f"at {spin_rpm:.0f} rpm"
        )

    angle_deg = solution.x[0]
    azimuth_deg = solution.x[1]

    endpoint = evaluate_endpoint_v7(
        angle_deg,
        azimuth_deg,
        spin_rpm
    )

    if endpoint is None:
        print(
            f"WARNING: endpoint evaluation failed "
            f"at {spin_rpm:.0f} rpm"
        )
        continue

    results.append({
        "spin_rpm": spin_rpm,
        "endpoint_angle_deg": angle_deg,
        "endpoint_azimuth_deg": azimuth_deg,
        "landing_x_m": endpoint["landing_x"],
        "landing_y_m": endpoint["landing_y"],
        "net_clearance_m": endpoint["clearance"]
    })

    # Continuation: use this solution as the
    # initial guess for the next spin value.
    previous_guess = solution.x


# --------------------------------------------------
# Create dataframe
# --------------------------------------------------

spin_azimuth_df = pd.DataFrame(results)


# --------------------------------------------------
# Display
# --------------------------------------------------

print("\nOuter admissibility boundary:")
print(
    spin_azimuth_df.to_string(
        index=False,
        formatters={
            "spin_rpm":
                lambda x: f"{x:.0f}",
            "endpoint_angle_deg":
                lambda x: f"{x:.6f}",
            "endpoint_azimuth_deg":
                lambda x: f"{x:.6f}",
            "landing_x_m":
                lambda x: f"{x:.6f}",
            "landing_y_m":
                lambda x: f"{x:.6f}",
            "net_clearance_m":
                lambda x: f"{x:.3e}"
        }
    )
)


# --------------------------------------------------
# Validation
# --------------------------------------------------

finite_check = np.all(
    np.isfinite(
        spin_azimuth_df[
            [
                "endpoint_angle_deg",
                "endpoint_azimuth_deg",
                "landing_x_m",
                "landing_y_m",
                "net_clearance_m"
            ]
        ].values
    )
)

width_check = np.all(
    spin_azimuth_df[
        "endpoint_azimuth_deg"
    ].values > 0
)

landing_width_check = np.max(
    np.abs(
        spin_azimuth_df[
            "landing_y_m"
        ].values
        - SERVICE_BOX_WIDTH
    )
) < 1e-5


print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(
    f"Finite endpoint solutions: "
    f"{'PASS' if finite_check else 'FAIL'}"
)

print(
    f"Positive azimuth endpoints: "
    f"{'PASS' if width_check else 'FAIL'}"
)

print(
    f"Landing y = service-box boundary: "
    f"{'PASS' if landing_width_check else 'FAIL'}"
)

print(
    "\nV7 Cell 12: SPIN × AZIMUTH ENVELOPE COMPLETE"
)

print("=" * 70)

In [ ]:
# V7 Cell 13 — Outer Envelope Visualization

import matplotlib.pyplot as plt

print("=" * 70)
print("V7 — OUTER SPIN × AZIMUTH ENVELOPE VISUALIZATION")
print("=" * 70)


spin = spin_azimuth_df["spin_rpm"].values

endpoint_azimuth = (
    spin_azimuth_df[
        "endpoint_azimuth_deg"
    ].values
)

endpoint_angle = (
    spin_azimuth_df[
        "endpoint_angle_deg"
    ].values
)

landing_x = (
    spin_azimuth_df[
        "landing_x_m"
    ].values
)


# --------------------------------------------------
# Figure 1 — Maximum admissible azimuth
# --------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    spin,
    endpoint_azimuth,
    marker="o"
)

plt.xlabel("Spin rate (rpm)")
plt.ylabel("Maximum admissible azimuth (degrees)")
plt.title(
    "Maximum Lateral Admissibility vs Spin"
)

plt.grid(True, alpha=0.3)

plt.show()


# --------------------------------------------------
# Figure 2 — Endpoint launch angle
# --------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    spin,
    endpoint_angle,
    marker="o"
)

plt.xlabel("Spin rate (rpm)")
plt.ylabel("Endpoint launch angle (degrees)")
plt.title(
    "Outer-Envelope Launch Angle vs Spin"
)

plt.grid(True, alpha=0.3)

plt.show()


# --------------------------------------------------
# Figure 3 — Landing position
# --------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    spin,
    landing_x,
    marker="o"
)

plt.axhline(
    SERVICE_LINE_X,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Spin rate (rpm)")
plt.ylabel("Landing x-position (m)")
plt.title(
    "Outer-Envelope Landing Position vs Spin"
)

plt.grid(True, alpha=0.3)

plt.show()


# --------------------------------------------------
# Quantitative summary
# --------------------------------------------------

print("\nKey changes across the tested spin range:")

print(
    f"Maximum azimuth:"
    f" {endpoint_azimuth[0]:.6f}°"
    f" → {endpoint_azimuth[-1]:.6f}°"
)

print(
    f"Total azimuth change:"
    f" {endpoint_azimuth[-1] - endpoint_azimuth[0]:+.6f}°"
)

print(
    f"Endpoint launch angle:"
    f" {endpoint_angle[0]:.6f}°"
    f" → {endpoint_angle[-1]:.6f}°"
)

print(
    f"Endpoint landing x:"
    f" {landing_x[0]:.6f} m"
    f" → {landing_x[-1]:.6f} m"
)


print("\n" + "=" * 70)
print("V7 Cell 13: VISUALIZATION COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 14 — Corrected Spin × Azimuth Admissibility Map

from scipy.optimize import brentq
import numpy as np
import pandas as pd

print("=" * 70)
print("V7 — CORRECTED SPIN × AZIMUTH ADMISSIBILITY MAP")
print("=" * 70)


# --------------------------------------------------
# Configuration
# --------------------------------------------------

SPEED_KMH = 200.0

SPIN_LEVELS = np.array([
    -2500,
    -1500,
    -500,
    0,
    500,
    1500,
    2500
], dtype=float)

AZIMUTH_LEVELS = np.arange(
    0.0,
    13.1,
    1.0
)

ANGLE_SCAN = np.linspace(
    -11.0,
    -5.0,
    25
)


# --------------------------------------------------
# Boundary evaluator
# --------------------------------------------------

def get_boundary_values_v7(
    angle_deg,
    azimuth_deg,
    spin_rpm
):

    result = evaluate_endpoint_v7(
        angle_deg,
        azimuth_deg,
        spin_rpm
    )

    if result is None:
        return np.nan, np.nan

    return (
        result["clearance"],
        result["landing_x"] - SERVICE_LINE_X
    )


# --------------------------------------------------
# Find all sign-change brackets
# --------------------------------------------------

def find_bracket_v7(
    function,
    angle_values
):

    values = []

    for angle in angle_values:

        value = function(angle)

        values.append(value)


    values = np.asarray(values)

    for i in range(
        len(angle_values) - 1
    ):

        f1 = values[i]
        f2 = values[i + 1]

        if (
            np.isfinite(f1)
            and np.isfinite(f2)
            and f1 * f2 <= 0
        ):

            return (
                angle_values[i],
                angle_values[i + 1]
            )

    return None


# --------------------------------------------------
# Solve one boundary
# --------------------------------------------------

def solve_global_boundary_v7(
    azimuth_deg,
    spin_rpm,
    boundary_type
):

    def function(angle):

        clearance, service_residual = (
            get_boundary_values_v7(
                angle,
                azimuth_deg,
                spin_rpm
            )
        )

        if boundary_type == "net":
            return clearance

        elif boundary_type == "service":
            return service_residual

        else:
            raise ValueError(
                "Unknown boundary type"
            )


    bracket = find_bracket_v7(
        function,
        ANGLE_SCAN
    )


    if bracket is None:
        return np.nan


    return brentq(
        function,
        bracket[0],
        bracket[1],
        xtol=1e-7
    )


# --------------------------------------------------
# Calculate map
# --------------------------------------------------

records = []


for spin_rpm in SPIN_LEVELS:

    print(
        f"\nProcessing spin = "
        f"{spin_rpm:.0f} rpm"
    )

    for azimuth_deg in AZIMUTH_LEVELS:

        net_boundary = (
            solve_global_boundary_v7(
                azimuth_deg,
                spin_rpm,
                "net"
            )
        )

        service_boundary = (
            solve_global_boundary_v7(
                azimuth_deg,
                spin_rpm,
                "service"
            )
        )


        # ------------------------------------------
        # Calculate admissible interval
        # ------------------------------------------

        if (
            np.isfinite(net_boundary)
            and np.isfinite(service_boundary)
        ):

            angle_width = (
                service_boundary
                - net_boundary
            )

            midpoint_angle = (
                0.5
                * (
                    net_boundary
                    + service_boundary
                )
            )

            midpoint = evaluate_endpoint_v7(
                midpoint_angle,
                azimuth_deg,
                spin_rpm
            )

            if midpoint is not None:

                midpoint_clearance = (
                    midpoint["clearance"]
                )

                midpoint_landing_x = (
                    midpoint["landing_x"]
                )

                midpoint_landing_y = (
                    midpoint["landing_y"]
                )

            else:

                midpoint_clearance = np.nan
                midpoint_landing_x = np.nan
                midpoint_landing_y = np.nan

        else:

            angle_width = np.nan
            midpoint_angle = np.nan
            midpoint_clearance = np.nan
            midpoint_landing_x = np.nan
            midpoint_landing_y = np.nan


        records.append({

            "spin_rpm":
                spin_rpm,

            "azimuth_deg":
                azimuth_deg,

            "net_boundary_deg":
                net_boundary,

            "service_boundary_deg":
                service_boundary,

            "angle_width_deg":
                angle_width,

            "midpoint_angle_deg":
                midpoint_angle,

            "midpoint_clearance_m":
                midpoint_clearance,

            "midpoint_landing_x_m":
                midpoint_landing_x,

            "midpoint_landing_y_m":
                midpoint_landing_y
        })


# --------------------------------------------------
# Dataframe
# --------------------------------------------------

spin_azimuth_map_df = pd.DataFrame(
    records
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n" + "=" * 70)
print("RESULT SUMMARY")
print("=" * 70)

total = len(
    spin_azimuth_map_df
)

valid = (
    spin_azimuth_map_df[
        "angle_width_deg"
    ].notna()
)

positive = (
    spin_azimuth_map_df[
        "angle_width_deg"
    ] > 0
)

print(
    f"Spin levels: {len(SPIN_LEVELS)}"
)

print(
    f"Azimuth levels: {len(AZIMUTH_LEVELS)}"
)

print(
    f"Total combinations: {total}"
)

print(
    f"Valid boundary solutions: "
    f"{valid.sum()} / {total}"
)

print(
    f"Positive-width regions: "
    f"{positive.sum()} / {total}"
)


# --------------------------------------------------
# Display
# --------------------------------------------------

print("\nAdmissibility map:")

print(
    spin_azimuth_map_df[
        [
            "spin_rpm",
            "azimuth_deg",
            "net_boundary_deg",
            "service_boundary_deg",
            "angle_width_deg"
        ]
    ].to_string(
        index=False,
        formatters={
            "spin_rpm":
                lambda x: f"{x:.0f}",

            "azimuth_deg":
                lambda x: f"{x:.1f}",

            "net_boundary_deg":
                lambda x:
                    f"{x:.6f}"
                    if np.isfinite(x)
                    else "NaN",

            "service_boundary_deg":
                lambda x:
                    f"{x:.6f}"
                    if np.isfinite(x)
                    else "NaN",

            "angle_width_deg":
                lambda x:
                    f"{x:.6f}"
                    if np.isfinite(x)
                    else "NaN"
        }
    )
)


# --------------------------------------------------
# Validation
# --------------------------------------------------

coverage = (
    valid.sum() / total
)

coverage_check = (
    coverage >= 0.90
)

positive_check = (
    positive[valid].all()
    if valid.sum() > 0
    else False
)


print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(
    f"Boundary coverage ≥ 90%: "
    f"{'PASS' if coverage_check else 'REVIEW'}"
)

print(
    f"All valid widths positive: "
    f"{'PASS' if positive_check else 'REVIEW'}"
)

print(
    "\nV7 Cell 14: CORRECTED MAP COMPLETE"
)

print("=" * 70)

In [ ]:
# V7 Cell 15 — Spin × Azimuth Admissibility Heatmap

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("V7 — SPIN × AZIMUTH ADMISSIBILITY HEATMAP")
print("=" * 70)


# --------------------------------------------------
# Convert dataframe to matrix
# --------------------------------------------------

pivot_width = spin_azimuth_map_df.pivot(
    index="spin_rpm",
    columns="azimuth_deg",
    values="angle_width_deg"
)

pivot_width = pivot_width.sort_index()


spin_values = pivot_width.index.values
azimuth_values = pivot_width.columns.values

width_matrix = pivot_width.values


# --------------------------------------------------
# Heatmap
# --------------------------------------------------

plt.figure(figsize=(10, 6))

image = plt.imshow(
    width_matrix,
    origin="lower",
    aspect="auto",
    extent=[
        azimuth_values.min(),
        azimuth_values.max(),
        spin_values.min(),
        spin_values.max()
    ],
    interpolation="nearest"
)

plt.colorbar(
    image,
    label="Admissible launch-angle width (degrees)"
)

plt.xlabel("Launch azimuth (degrees)")
plt.ylabel("Spin rate (rpm)")

plt.title(
    "Spin × Azimuth Admissibility Envelope"
)

plt.show()


# --------------------------------------------------
# Quantitative extrema
# --------------------------------------------------

max_index = np.unravel_index(
    np.nanargmax(width_matrix),
    width_matrix.shape
)

min_index = np.unravel_index(
    np.nanargmin(width_matrix),
    width_matrix.shape
)


max_width = width_matrix[max_index]
min_width = width_matrix[min_index]

max_spin = spin_values[max_index[0]]
max_azimuth = azimuth_values[max_index[1]]

min_spin = spin_values[min_index[0]]
min_azimuth = azimuth_values[min_index[1]]


print("\nEnvelope extrema:")

print(
    f"Maximum width: "
    f"{max_width:.6f}°"
)

print(
    f"  Spin: "
    f"{max_spin:.0f} rpm"
)

print(
    f"  Azimuth: "
    f"{max_azimuth:.1f}°"
)

print(
    f"\nMinimum width: "
    f"{min_width:.6f}°"
)

print(
    f"  Spin: "
    f"{min_spin:.0f} rpm"
)

print(
    f"  Azimuth: "
    f"{min_azimuth:.1f}°"
)


# --------------------------------------------------
# Spin effect at fixed azimuth
# --------------------------------------------------

for azimuth in [0.0, 5.0, 10.0, 13.0]:

    row = spin_azimuth_map_df[
        np.isclose(
            spin_azimuth_map_df[
                "azimuth_deg"
            ],
            azimuth
        )
    ]

    width_range = (
        row["angle_width_deg"].max()
        - row["angle_width_deg"].min()
    )

    print(
        f"\nAzimuth {azimuth:.1f}°:"
    )

    print(
        f"  Width range across spin: "
        f"{width_range:.6f}°"
    )


print("\n" + "=" * 70)
print("V7 Cell 15: HEATMAP COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 16 — Boundary Branch Consistency Check

from scipy.optimize import brentq
import numpy as np

print("=" * 70)
print("V7 — BOUNDARY BRANCH CONSISTENCY CHECK")
print("=" * 70)


# --------------------------------------------------
# Test spins
# --------------------------------------------------

TEST_SPINS = np.array([
    -2500,
    -1500,
    0,
    1500,
    2500
], dtype=float)


AZIMUTH_TEST = 0.0


# --------------------------------------------------
# Robust local root finder
# --------------------------------------------------

def local_root_v7(
    function,
    center,
    initial_half_width=0.10,
    max_half_width=1.50
):

    width = initial_half_width

    while width <= max_half_width:

        left = center - width
        right = center + width

        f_left = function(left)
        f_right = function(right)

        if (
            np.isfinite(f_left)
            and np.isfinite(f_right)
            and f_left * f_right <= 0
        ):

            return brentq(
                function,
                left,
                right,
                xtol=1e-8
            )

        width *= 1.5

    return np.nan


# --------------------------------------------------
# Solve around the VERIFIED Cell 6 branch
# --------------------------------------------------

comparison = []


for spin_rpm in TEST_SPINS:

    # ----------------------------------------------
    # Retrieve verified Cell 6 boundary
    # ----------------------------------------------

    reference = spin_boundary_df[
        np.isclose(
            spin_boundary_df["spin_rpm"],
            spin_rpm
        )
    ]

    if len(reference) != 1:

        print(
            f"Missing Cell 6 reference "
            f"for {spin_rpm:.0f} rpm"
        )

        continue


    reference_net = float(
        reference[
            "net_boundary_deg"
        ].iloc[0]
    )

    reference_service = float(
        reference[
            "service_boundary_deg"
        ].iloc[0]
    )


    # ----------------------------------------------
    # Net boundary
    # ----------------------------------------------

    def net_function(angle):

        result = evaluate_endpoint_v7(
            angle,
            AZIMUTH_TEST,
            spin_rpm
        )

        if result is None:
            return np.nan

        return result["clearance"]


    net_boundary = local_root_v7(
        net_function,
        reference_net
    )


    # ----------------------------------------------
    # Service boundary
    # ----------------------------------------------

    def service_function(angle):

        result = evaluate_endpoint_v7(
            angle,
            AZIMUTH_TEST,
            spin_rpm
        )

        if result is None:
            return np.nan

        return (
            result["landing_x"]
            - SERVICE_LINE_X
        )


    service_boundary = local_root_v7(
        service_function,
        reference_service
    )


    # ----------------------------------------------
    # Compare
    # ----------------------------------------------

    net_error = (
        net_boundary
        - reference_net
    )

    service_error = (
        service_boundary
        - reference_service
    )


    width = (
        service_boundary
        - net_boundary
    )


    reference_width = (
        reference_service
        - reference_net
    )


    width_error = (
        width
        - reference_width
    )


    comparison.append({

        "spin_rpm":
            spin_rpm,

        "reference_net_deg":
            reference_net,

        "recovered_net_deg":
            net_boundary,

        "net_error_deg":
            net_error,

        "reference_service_deg":
            reference_service,

        "recovered_service_deg":
            service_boundary,

        "service_error_deg":
            service_error,

        "reference_width_deg":
            reference_width,

        "recovered_width_deg":
            width,

        "width_error_deg":
            width_error
    })


# --------------------------------------------------
# Display
# --------------------------------------------------

comparison_df = pd.DataFrame(
    comparison
)


print(
    comparison_df.to_string(
        index=False,
        formatters={

            "spin_rpm":
                lambda x:
                    f"{x:.0f}",

            "reference_net_deg":
                lambda x:
                    f"{x:.6f}",

            "recovered_net_deg":
                lambda x:
                    f"{x:.6f}",

            "net_error_deg":
                lambda x:
                    f"{x:.3e}",

            "reference_service_deg":
                lambda x:
                    f"{x:.6f}",

            "recovered_service_deg":
                lambda x:
                    f"{x:.6f}",

            "service_error_deg":
                lambda x:
                    f"{x:.3e}",

            "reference_width_deg":
                lambda x:
                    f"{x:.6f}",

            "recovered_width_deg":
                lambda x:
                    f"{x:.6f}",

            "width_error_deg":
                lambda x:
                    f"{x:.3e}"
        }
    )
)


# --------------------------------------------------
# Validation
# --------------------------------------------------

max_net_error = np.max(
    np.abs(
        comparison_df[
            "net_error_deg"
        ]
    )
)

max_service_error = np.max(
    np.abs(
        comparison_df[
            "service_error_deg"
        ]
    )
)

max_width_error = np.max(
    np.abs(
        comparison_df[
            "width_error_deg"
        ]
    )
)


print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(
    f"Maximum net-boundary error: "
    f"{max_net_error:.3e}°"
)

print(
    f"Maximum service-boundary error: "
    f"{max_service_error:.3e}°"
)

print(
    f"Maximum width error: "
    f"{max_width_error:.3e}°"
)


branch_check = (
    max_net_error < 1e-5
    and
    max_service_error < 1e-5
    and
    max_width_error < 1e-5
)


print(
    f"\nVerified Cell 6 branch recovered: "
    f"{'PASS' if branch_check else 'FAIL'}"
)

print(
    "\nV7 Cell 16: BRANCH CONSISTENCY CHECK COMPLETE"
)

print("=" * 70)

In [ ]:
# V7 Cell 17 — Spin Boundary Diagnostic

print("=" * 70)
print("V7 — SPIN BOUNDARY DIAGNOSTIC")
print("=" * 70)


# --------------------------------------------------
# Cell 6 reference boundaries
# --------------------------------------------------

reference_data = {
    -2500: -10.312723,
    -1500:  -9.769817,
    -500:   -9.087440,
       0:   -8.675886,
     500:   -8.264235,
    1000:   -7.902207,
    1500:   -7.581312,
    2000:   -7.294897,
    2500:   -7.037675
}


# --------------------------------------------------
# Evaluate clearance around each reference
# --------------------------------------------------

for spin_rpm, reference_angle in reference_data.items():

    print("\n" + "-" * 70)
    print(
        f"Spin: {spin_rpm:+.0f} rpm"
    )
    print(
        f"Cell 6 reference angle: "
        f"{reference_angle:.6f}°"
    )
    print("-" * 70)

    test_angles = [
        reference_angle - 0.20,
        reference_angle - 0.10,
        reference_angle,
        reference_angle + 0.10,
        reference_angle + 0.20
    ]

    for angle in test_angles:

        result = evaluate_endpoint_v7(
            angle,
            0.0,
            spin_rpm
        )

        if result is None:

            print(
                f"  angle {angle: .6f}°"
                f"  → evaluation FAILED"
            )

        else:

            print(
                f"  angle {angle: .6f}°"
                f"  → clearance = "
                f"{result['clearance']:+.8f} m"
            )


print("\n" + "=" * 70)
print("V7 Cell 17: DIAGNOSTIC COMPLETE")
print("=" * 70)

In [ ]:
# V7 Cell 18 — Correct Spin Conversion and Branch Verification

print("=" * 70)
print("V7 — CORRECTED SPIN CONVERSION VERIFICATION")
print("=" * 70)


# --------------------------------------------------
# Corrected trajectory evaluator
# --------------------------------------------------

def evaluate_endpoint_v7(
    launch_angle_deg,
    azimuth_deg,
    spin_rpm
):
    """
    Evaluate net clearance and actual ground landing.

    IMPORTANT:
    RPM is converted to rad/s using

        omega = rpm * 2*pi / 60

    """

    speed_ms = SPEED_KMH / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )

    vx0 = (
        speed_ms
        * np.cos(theta)
        * np.cos(phi)
    )

    vy0 = (
        speed_ms
        * np.cos(theta)
        * np.sin(phi)
    )

    vz0 = (
        speed_ms
        * np.sin(theta)
    )

    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT,
        vx0,
        vy0,
        vz0
    ])


    # ----------------------------------------------
    # CORRECT RPM → rad/s conversion
    # ----------------------------------------------

    spin_rad_s = (
        spin_rpm
        * 2.0
        * np.pi
        / 60.0
    )

    omega = np.array([
        0.0,
        spin_rad_s,
        0.0
    ])


    # ----------------------------------------------
    # Net event
    # ----------------------------------------------

    def net_event(t, state):
        return state[0] - NET_X

    net_event.terminal = True
    net_event.direction = 1


    solution_net = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),
        (0.0, 2.0),
        initial_state,
        events=net_event,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.002
    )


    if len(solution_net.t_events[0]) == 0:
        return None


    net_state = (
        solution_net.y_events[0][0]
    )

    net_y = net_state[1]
    net_z = net_state[2]

    clearance = (
        net_z
        - net_height(net_y)
    )


    # ----------------------------------------------
    # Continue to ground
    # ----------------------------------------------

    def ground_event(t, state):
        return state[2]

    ground_event.terminal = True
    ground_event.direction = -1


    solution_ground = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),
        (
            solution_net.t_events[0][0],
            2.0
        ),
        net_state,
        events=ground_event,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.002
    )


    if len(solution_ground.t_events[0]) == 0:
        return None


    landing_state = (
        solution_ground.y_events[0][0]
    )


    return {
        "clearance": clearance,
        "landing_x": landing_state[0],
        "landing_y": landing_state[1],
        "landing_z": landing_state[2]
    }


# --------------------------------------------------
# Cell 6 reference boundaries
# --------------------------------------------------

reference_data = {
    -2500: (-10.312723, -9.751799),
    -1500: ( -9.769817, -8.897104),
     -500: ( -9.087440, -7.812768),
        0: ( -8.675886, -7.152987),
      500: ( -8.264235, -6.492758),
     1500: ( -7.581312, -5.405918),
     2500: ( -7.037675, -4.547922)
}


# --------------------------------------------------
# Verify net boundaries
# --------------------------------------------------

print("\nNet-boundary verification:")

net_errors = []

for spin_rpm, (reference_net, _) in reference_data.items():

    result = evaluate_endpoint_v7(
        reference_net,
        0.0,
        spin_rpm
    )

    if result is None:

        print(
            f"{spin_rpm:+.0f} rpm → FAILED"
        )

        continue

    error = result["clearance"]

    net_errors.append(
        abs(error)
    )

    print(
        f"{spin_rpm:+.0f} rpm"
        f"  angle = {reference_net:.6f}°"
        f"  clearance = {error:+.3e} m"
    )


# --------------------------------------------------
# Verify service boundaries
# --------------------------------------------------

print("\nService-boundary verification:")

service_errors = []

for spin_rpm, (_, reference_service) in reference_data.items():

    result = evaluate_endpoint_v7(
        reference_service,
        0.0,
        spin_rpm
    )

    if result is None:

        print(
            f"{spin_rpm:+.0f} rpm → FAILED"
        )

        continue

    error = (
        result["landing_x"]
        - SERVICE_LINE_X
    )

    service_errors.append(
        abs(error)
    )

    print(
        f"{spin_rpm:+.0f} rpm"
        f"  angle = {reference_service:.6f}°"
        f"  landing-x error = {error:+.3e} m"
    )


# --------------------------------------------------
# Validation
# --------------------------------------------------

max_net_error = max(
    net_errors
)

max_service_error = max(
    service_errors
)


print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(
    f"Maximum net-boundary residual: "
    f"{max_net_error:.3e} m"
)

print(
    f"Maximum service-boundary residual: "
    f"{max_service_error:.3e} m"
)


verification_pass = (
    max_net_error < 1e-4
    and
    max_service_error < 1e-4
)


print(
    f"\nCell 6 branch reproduced: "
    f"{'PASS' if verification_pass else 'FAIL'}"
)

print(
    "\nV7 Cell 18: SPIN CONVERSION FIX VERIFIED"
)

print("=" * 70)

In [ ]:
# V7 Cell 19 — Corrected Spin × Azimuth Envelope

from scipy.optimize import brentq
import numpy as np
import pandas as pd

print("=" * 70)
print("V7 — CORRECTED SPIN × AZIMUTH ENVELOPE")
print("=" * 70)


# --------------------------------------------------
# Configuration
# --------------------------------------------------

SPIN_LEVELS = np.array([
    -2500,
    -1500,
    -500,
    0,
    500,
    1500,
    2500
], dtype=float)

AZIMUTH_LEVELS = np.arange(
    0.0,
    13.1,
    1.0
)

ANGLE_SCAN = np.linspace(
    -12.0,
    -3.0,
    37
)


# --------------------------------------------------
# Boundary functions
# --------------------------------------------------

def net_residual_v7(
    angle_deg,
    azimuth_deg,
    spin_rpm
):

    result = evaluate_endpoint_v7(
        angle_deg,
        azimuth_deg,
        spin_rpm
    )

    if result is None:
        return np.nan

    return result["clearance"]


def service_residual_v7(
    angle_deg,
    azimuth_deg,
    spin_rpm
):

    result = evaluate_endpoint_v7(
        angle_deg,
        azimuth_deg,
        spin_rpm
    )

    if result is None:
        return np.nan

    return (
        result["landing_x"]
        - SERVICE_LINE_X
    )


# --------------------------------------------------
# Find all brackets
# --------------------------------------------------

def find_all_brackets_v7(
    function,
    angle_values
):

    brackets = []

    values = []

    for angle in angle_values:

        values.append(
            function(angle)
        )

    values = np.asarray(values)

    for i in range(
        len(angle_values) - 1
    ):

        f1 = values[i]
        f2 = values[i + 1]

        if (
            np.isfinite(f1)
            and np.isfinite(f2)
            and f1 * f2 <= 0
        ):

            brackets.append(
                (
                    angle_values[i],
                    angle_values[i + 1]
                )
            )

    return brackets


# --------------------------------------------------
# Select root nearest validated branch
# --------------------------------------------------

def solve_branch_root_v7(
    function,
    angle_center,
    angle_values
):

    brackets = find_all_brackets_v7(
        function,
        angle_values
    )

    if len(brackets) == 0:
        return np.nan

    roots = []

    for left, right in brackets:

        try:

            root = brentq(
                function,
                left,
                right,
                xtol=1e-8
            )

            roots.append(root)

        except ValueError:

            pass


    if len(roots) == 0:
        return np.nan


    roots = np.asarray(roots)

    # Select the root closest to the
    # previously validated branch.
    return roots[
        np.argmin(
            np.abs(
                roots
                - angle_center
            )
        )
    ]


# --------------------------------------------------
# Build corrected envelope
# --------------------------------------------------

records = []


for spin_rpm in SPIN_LEVELS:

    print(
        f"\nProcessing "
        f"{spin_rpm:+.0f} rpm"
    )


    # Retrieve verified zero-azimuth
    # Cell 6 boundaries.
    reference = spin_boundary_df[
        np.isclose(
            spin_boundary_df[
                "spin_rpm"
            ],
            spin_rpm
        )
    ]


    if len(reference) != 1:

        raise RuntimeError(
            f"Missing Cell 6 reference "
            f"for {spin_rpm:.0f} rpm"
        )


    previous_net = float(
        reference[
            "net_boundary_deg"
        ].iloc[0]
    )

    previous_service = float(
        reference[
            "service_boundary_deg"
        ].iloc[0]
    )


    for azimuth_deg in AZIMUTH_LEVELS:

        # ------------------------------------------
        # Net boundary
        # ------------------------------------------

        net_boundary = (
            solve_branch_root_v7(
                lambda angle:
                    net_residual_v7(
                        angle,
                        azimuth_deg,
                        spin_rpm
                    ),
                previous_net,
                ANGLE_SCAN
            )
        )


        # ------------------------------------------
        # Service boundary
        # ------------------------------------------

        service_boundary = (
            solve_branch_root_v7(
                lambda angle:
                    service_residual_v7(
                        angle,
                        azimuth_deg,
                        spin_rpm
                    ),
                previous_service,
                ANGLE_SCAN
            )
        )


        # ------------------------------------------
        # Width
        # ------------------------------------------

        if (
            np.isfinite(net_boundary)
            and
            np.isfinite(service_boundary)
        ):

            width = (
                service_boundary
                - net_boundary
            )

        else:

            width = np.nan


        records.append({

            "spin_rpm":
                spin_rpm,

            "azimuth_deg":
                azimuth_deg,

            "net_boundary_deg":
                net_boundary,

            "service_boundary_deg":
                service_boundary,

            "angle_width_deg":
                width
        })


        # Continue along the branch.
        if np.isfinite(net_boundary):
            previous_net = net_boundary

        if np.isfinite(service_boundary):
            previous_service = service_boundary


# --------------------------------------------------
# Dataframe
# --------------------------------------------------

corrected_map_df = pd.DataFrame(
    records
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

total = len(
    corrected_map_df
)

valid = (
    corrected_map_df[
        "angle_width_deg"
    ].notna()
)

positive = (
    corrected_map_df[
        "angle_width_deg"
    ] > 0
)


print("\n" + "=" * 70)
print("RESULT SUMMARY")
print("=" * 70)

print(
    f"Total combinations: "
    f"{total}"
)

print(
    f"Valid boundaries: "
    f"{valid.sum()} / {total}"
)

print(
    f"Positive-width regions: "
    f"{positive.sum()} / {total}"
)


# --------------------------------------------------
# Critical zero-azimuth regression
# --------------------------------------------------

print("\nZero-azimuth regression:")

zero_azimuth = corrected_map_df[
    np.isclose(
        corrected_map_df[
            "azimuth_deg"
        ],
        0.0
    )
]


for _, row in zero_azimuth.iterrows():

    spin = row["spin_rpm"]

    reference = spin_boundary_df[
        np.isclose(
            spin_boundary_df[
                "spin_rpm"
            ],
            spin
        )
    ].iloc[0]

    net_error = (
        row["net_boundary_deg"]
        - reference[
            "net_boundary_deg"
        ]
    )

    service_error = (
        row["service_boundary_deg"]
        - reference[
            "service_boundary_deg"
        ]
    )

    print(
        f"{spin:+.0f} rpm:"
        f"  net error = {net_error:+.3e}°"
        f"  service error = {service_error:+.3e}°"
    )


# --------------------------------------------------
# Validation
# --------------------------------------------------

max_net_error = np.max(
    np.abs(
        zero_azimuth[
            "net_boundary_deg"
        ].values
        -
        spin_boundary_df[
            "net_boundary_deg"
        ].values
    )
)

max_service_error = np.max(
    np.abs(
        zero_azimuth[
            "service_boundary_deg"
        ].values
        -
        spin_boundary_df[
            "service_boundary_deg"
        ].values
    )
)


coverage_pass = (
    valid.sum()
    == total
)

positive_pass = (
    positive.all()
)

regression_pass = (
    max_net_error < 1e-5
    and
    max_service_error < 1e-5
)


print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(
    f"100% boundary coverage: "
    f"{'PASS' if coverage_pass else 'FAIL'}"
)

print(
    f"All widths positive: "
    f"{'PASS' if positive_pass else 'FAIL'}"
)

print(
    f"Zero-azimuth regression: "
    f"{'PASS' if regression_pass else 'FAIL'}"
)

print(
    "\nV7 Cell 19: CORRECTED ENVELOPE COMPLETE"
)

print("=" * 70)

In [ ]:
# V7 Cell 20 — Final Validation of Corrected Envelope

print("=" * 70)
print("V7 — FINAL CORRECTED ENVELOPE VALIDATION")
print("=" * 70)


# --------------------------------------------------
# Zero-azimuth rows from corrected map
# --------------------------------------------------

zero_azimuth = corrected_map_df[
    np.isclose(
        corrected_map_df["azimuth_deg"],
        0.0
    )
].copy()


# --------------------------------------------------
# Match against the corresponding Cell 6
# reference rows by spin
# --------------------------------------------------

reference_zero = spin_boundary_df[
    spin_boundary_df["spin_rpm"].isin(
        zero_azimuth["spin_rpm"]
    )
].copy()


zero_azimuth = zero_azimuth.sort_values(
    "spin_rpm"
)

reference_zero = reference_zero.sort_values(
    "spin_rpm"
)


# --------------------------------------------------
# Calculate errors
# --------------------------------------------------

net_errors = (
    zero_azimuth[
        "net_boundary_deg"
    ].values
    -
    reference_zero[
        "net_boundary_deg"
    ].values
)


service_errors = (
    zero_azimuth[
        "service_boundary_deg"
    ].values
    -
    reference_zero[
        "service_boundary_deg"
    ].values
)


max_net_error = np.max(
    np.abs(net_errors)
)

max_service_error = np.max(
    np.abs(service_errors)
)


# --------------------------------------------------
# Coverage and positivity
# --------------------------------------------------

total = len(corrected_map_df)

valid = corrected_map_df[
    [
        "net_boundary_deg",
        "service_boundary_deg",
        "angle_width_deg"
    ]
].notna().all(axis=1)

positive = (
    corrected_map_df[
        "angle_width_deg"
    ] > 0
)


# --------------------------------------------------
# Print detailed regression
# --------------------------------------------------

print("\nZero-azimuth regression:")

for i in range(
    len(zero_azimuth)
):

    spin = zero_azimuth[
        "spin_rpm"
    ].iloc[i]

    print(
        f"{spin:+.0f} rpm:"
        f"  net error = "
        f"{net_errors[i]:+.3e}°"
        f"  service error = "
        f"{service_errors[i]:+.3e}°"
    )


# --------------------------------------------------
# Final validation
# --------------------------------------------------

coverage_pass = (
    valid.sum() == total
)

positive_pass = (
    positive.all()
)

regression_pass = (
    max_net_error < 1e-5
    and
    max_service_error < 1e-5
)


print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print(
    f"Boundary coverage: "
    f"{valid.sum()} / {total} "
    f"→ "
    f"{'PASS' if coverage_pass else 'FAIL'}"
)

print(
    f"Positive-width regions: "
    f"{positive.sum()} / {total} "
    f"→ "
    f"{'PASS' if positive_pass else 'FAIL'}"
)

print(
    f"Maximum net-boundary error: "
    f"{max_net_error:.3e}°"
)

print(
    f"Maximum service-boundary error: "
    f"{max_service_error:.3e}°"
)

print(
    f"Zero-azimuth regression: "
    f"{'PASS' if regression_pass else 'FAIL'}"
)


overall_pass = (
    coverage_pass
    and positive_pass
    and regression_pass
)

print("\n" + "=" * 70)

print(
    f"V7 CORRECTED SPIN × AZIMUTH ENVELOPE: "
    f"{'PASS' if overall_pass else 'FAIL'}"
)

print("=" * 70)

In [ ]:
# V7 Cell 21 — Corrected Spin × Azimuth Envelope Heatmap

import matplotlib.pyplot as plt

pivot = corrected_map_df.pivot(
    index="spin_rpm",
    columns="azimuth_deg",
    values="angle_width_deg"
)

plt.figure(figsize=(10, 6))

plt.imshow(
    pivot.values,
    aspect="auto",
    origin="lower",
    extent=[
        pivot.columns.min(),
        pivot.columns.max(),
        pivot.index.min(),
        pivot.index.max()
    ]
)

plt.colorbar(label="Admissible launch-angle width (degrees)")

plt.xlabel("Launch azimuth (degrees)")
plt.ylabel("Spin rate (rpm)")
plt.title(
    "Serve Admissibility Width Across Spin and Launch Azimuth"
)

plt.tight_layout()
plt.show()